# Train SelTDA teacher trên AdVQA (Kaggle)

Notebook clone SelTDA, tải AdVQA và các ảnh COCO val2014 được tham chiếu, chuyển dữ liệu sang schema VQG rồi fine-tune BLIP teacher.

> AdVQA chính thức chỉ phát hành **validation labels** (10.000 câu/100.000 answers) và test không có labels. Vì vậy notebook dùng labeled validation split, mặc định giữ lại 10% làm internal holdout. Đây không còn là protocol đánh giá zero-shot AdVQA chính thức.

In [ ]:
from pathlib import Path

REPO_URL = 'https://github.com/fantastichaha11/SelTDA.git'
BRANCH = 'feat/pseudo-label-filter'
REPO_DIR = Path('/kaggle/working/SelTDA')
DATA_ROOT = Path('/kaggle/working/data')
ADVQA_ROOT = DATA_ROOT / 'advqa'
COCO_ROOT = DATA_ROOT / 'coco2014'
OUTPUT_DIR = Path('/kaggle/working/advqa_teacher')
CONFIG_PATH = REPO_DIR / 'configs/advqg_kaggle.yaml'

MAX_GPUS = 2
BATCH_SIZE_PER_GPU = 8
MAX_EPOCH = 5
HOLDOUT_FRACTION = 0.10
SEED = 42
MAX_RECORDS = None  # Ví dụ 256 để smoke test; None để dùng toàn bộ
IMAGE_DOWNLOAD_WORKERS = 24

In [ ]:
import subprocess, sys

if not REPO_DIR.exists():
    subprocess.check_call(['git', 'clone', '--depth', '1', '--branch', BRANCH, REPO_URL, str(REPO_DIR)])
else:
    print('Repo đã tồn tại, bỏ qua clone:', REPO_DIR)

# Không cài requirements.txt vì pin NumPy cũ không tương thích Python 3.12 của Kaggle.
packages = [
    'omegaconf==2.3.0',
    'hydra-core==1.3.2',
    'timm==0.4.12',
    'fairscale==0.4.13',
    'transformers==4.36.1',
]
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', *packages])
print('Clone và dependency setup hoàn tất.')

In [ ]:
import json, os, random, time
import requests
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm

ADVQA_ROOT.mkdir(parents=True, exist_ok=True)
(COCO_ROOT / 'val2014').mkdir(parents=True, exist_ok=True)

QUESTIONS_URL = 'https://dl.fbaipublicfiles.com/advqa/v1_OpenEnded_mscoco_val2017_advqa_questions.json'
ANNOTATIONS_URL = 'https://dl.fbaipublicfiles.com/advqa/v1_mscoco_val2017_advqa_annotations.json'
QUESTIONS_PATH = ADVQA_ROOT / 'v1_OpenEnded_mscoco_val2017_advqa_questions.json'
ANNOTATIONS_PATH = ADVQA_ROOT / 'v1_mscoco_val2017_advqa_annotations.json'

def download_file(url, destination):
    if destination.is_file() and destination.stat().st_size > 0:
        return
    tmp = destination.with_suffix(destination.suffix + '.tmp')
    with requests.get(url, stream=True, timeout=120) as response:
        response.raise_for_status()
        with tmp.open('wb') as f:
            for chunk in response.iter_content(1024 * 1024):
                if chunk:
                    f.write(chunk)
    os.replace(tmp, destination)

download_file(QUESTIONS_URL, QUESTIONS_PATH)
download_file(ANNOTATIONS_URL, ANNOTATIONS_PATH)
with QUESTIONS_PATH.open() as f:
    raw_questions = json.load(f)['questions']
with ANNOTATIONS_PATH.open() as f:
    raw_annotations = json.load(f)['annotations']
print(f'Questions={len(raw_questions):,}; annotations={len(raw_annotations):,}')

In [ ]:
annotation_by_qid = {int(a['question_id']): a for a in raw_annotations}
questions = [q for q in raw_questions if int(q['question_id']) in annotation_by_qid]
if MAX_RECORDS is not None:
    questions = questions[:MAX_RECORDS]
image_ids = sorted({int(q['image_id']) for q in questions})

def coco_filename(image_id):
    return f'COCO_val2014_{image_id:012d}.jpg'

def download_coco_image(image_id):
    filename = coco_filename(image_id)
    destination = COCO_ROOT / 'val2014' / filename
    if destination.is_file() and destination.stat().st_size > 0:
        return destination
    url = f'https://images.cocodataset.org/val2014/{filename}'
    last_error = None
    for attempt in range(4):
        try:
            response = requests.get(url, timeout=60)
            response.raise_for_status()
            tmp = destination.with_suffix('.jpg.tmp')
            tmp.write_bytes(response.content)
            os.replace(tmp, destination)
            return destination
        except Exception as error:
            last_error = error
            time.sleep(2 ** attempt)
    raise RuntimeError(f'Không tải được COCO image {image_id}: {last_error}')

missing = [i for i in image_ids if not (COCO_ROOT / 'val2014' / coco_filename(i)).is_file()]
print(f'Ảnh cần dùng={len(image_ids):,}; cần tải={len(missing):,}')
with ThreadPoolExecutor(max_workers=IMAGE_DOWNLOAD_WORKERS) as pool:
    futures = [pool.submit(download_coco_image, image_id) for image_id in missing]
    for future in tqdm(as_completed(futures), total=len(futures), desc='COCO val2014'):
        future.result()
print('Tải ảnh hoàn tất.')

In [ ]:
records = []
for question in questions:
    qid = int(question['question_id'])
    annotation = annotation_by_qid[qid]
    answers = [item['answer'].strip() for item in annotation['answers'] if item.get('answer', '').strip()]
    if not answers:
        continue
    image_id = int(question['image_id'])
    records.append({
        'dataset': 'advqa',
        'image': f'val2014/{coco_filename(image_id)}',
        'question': question['question'].strip(),
        'question_id': qid,
        'answer': answers,
    })

rng = random.Random(SEED)
rng.shuffle(records)
holdout_size = int(len(records) * HOLDOUT_FRACTION)
holdout_records = records[:holdout_size] if holdout_size else records[:min(1000, len(records))]
train_records = records[holdout_size:] if holdout_size else records
answer_list = sorted({answer for record in records for answer in record['answer']})

def write_json(path, data):
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + '.tmp')
    with tmp.open('w', encoding='utf-8') as f:
        json.dump(data, f, ensure_ascii=False)
    os.replace(tmp, path)

write_json(ADVQA_ROOT / 'train.json', train_records)
write_json(ADVQA_ROOT / 'test.json', holdout_records)
write_json(ADVQA_ROOT / 'val.json', holdout_records)
write_json(ADVQA_ROOT / 'answer_list.json', answer_list)
print(f'Train={len(train_records):,}; holdout={len(holdout_records):,}; answers={len(answer_list):,}')
assert all((COCO_ROOT / record['image']).is_file() for record in records)

In [ ]:
# Thêm AdVQA vào generic AokVqgDataset router của repo clone.
data_init = REPO_DIR / 'data/__init__.py'
source = data_init.read_text()
old = '("aokvqa", "okvqa", "artvqa", "pathvqa")'
new = '("aokvqa", "okvqa", "artvqa", "pathvqa", "advqa")'
if old in source:
    data_init.write_text(source.replace(old, new, 1))
elif '"advqa"' not in source:
    raise RuntimeError('Không tìm thấy dataset router cần patch')

import yaml
config = {
    'vqa_root': str(COCO_ROOT),
    'vg_root': None,
    'train_files': ['train'],
    'ann_root': str(ADVQA_ROOT),
    'dataset_name': 'advqa',
    'truncate_train_dataset_to': None,
    'append_rationale_to_answer': False,
    'append_rationale_to_question': False,
    'tokenizer_max_length': 40,
    'use_rationale': False,
    'generate_rationale_first': False,
    'pretrained': 'https://storage.googleapis.com/sfr-vision-language-research/BLIP/models/model_base_caption_capfilt_large.pth',
    'use_validation_set_as_test_set': False,
    'vit': 'base',
    'vit_grad_ckpt': True,
    'vit_ckpt_layer': 4,
    'batch_size': BATCH_SIZE_PER_GPU,
    'init_lr': 1e-5,
    'image_size': 384,
    'max_length': 40,
    'min_length': 5,
    'num_beams': 3,
    'prompt': '',
    'weight_decay': 0.05,
    'min_lr': 0,
    'max_epoch': MAX_EPOCH,
    'torch_home': '/kaggle/working/torch_home',
    'wandb': False,
    'save_last_only': True,
}
CONFIG_PATH.write_text(yaml.safe_dump(config, sort_keys=False))
print(CONFIG_PATH.read_text())

In [ ]:
import torch

assert torch.cuda.is_available(), 'Bật GPU trong Kaggle Settings > Accelerator'
gpu_count = min(MAX_GPUS, torch.cuda.device_count())
for gpu_id in range(gpu_count):
    print(f'cuda:{gpu_id}:', torch.cuda.get_device_name(gpu_id))

smoke = subprocess.run(
    [sys.executable, '-c', 'import train_vqg; print("train_vqg import OK")'],
    cwd=REPO_DIR, text=True, capture_output=True
)
print(smoke.stdout)
if smoke.returncode != 0:
    raise RuntimeError(smoke.stderr)

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
command = [
    sys.executable, '-m', 'torch.distributed.run',
    '--standalone', f'--nproc_per_node={gpu_count}',
    'train_vqg.py',
    '--config=configs/advqg_kaggle.yaml',
    f'--output_dir={OUTPUT_DIR}',
    '--no-resume',
]
print(' '.join(map(str, command)))
subprocess.check_call(command, cwd=REPO_DIR)
print('Training hoàn tất.')

In [ ]:
checkpoints = sorted(OUTPUT_DIR.glob('checkpoint_*.pth'))
assert checkpoints, f'Không tìm thấy checkpoint trong {OUTPUT_DIR}'
for path in checkpoints:
    print(f'{path.name}: {path.stat().st_size / 1024**3:.2f} GiB')
print('Teacher checkpoint:', checkpoints[-1])
print('Config:', OUTPUT_DIR / 'config.yaml')
print('Log:', OUTPUT_DIR / 'log.txt')